# Utforsking og figurar

Denne notatboka er berre for å **sjå på data og resultat**. All logikk ligg i `src/`, så her blir ingenting trena.

Rekkjefølgje (frå rotmappa i repoet):

```bash
python scripts/download_data.py
python src/prepare_crops.py
python src/train.py          # lagar models/ og reports/
```

Del 1 kan køyrast så snart utklippa finst. Del 2 krev at `train.py` er køyrd.

Tips for samarbeid: tøm output før commit (*Kernel → Restart & Clear Output*), så blir det færre Git-konfliktar.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from dataset import DATA, RANDOM_STATE, load_meta, load_features, labels_and_groups, group_split

REPORTS = ROOT / "reports"

# Del 1: Datasettet

In [ ]:
meta = load_meta()
y, groups = labels_and_groups(meta)
print(meta.shape)
print(meta["label"].value_counts())
print("\nTal foto:", meta["group"].nunique())
meta.head()

## Døme frå kvar klasse

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, label in enumerate(["ripe", "unripe"]):
    files = meta.loc[meta["label"] == label, "file"].to_numpy()
    for ax, f in zip(axes[row], rng.choice(files, size=min(8, len(files)), replace=False)):
        ax.imshow(Image.open(DATA / f))
        ax.set_title(label, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Storleik på utklippa

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for label in ["ripe", "unripe"]:
    ax.hist(meta.loc[meta["label"] == label, "width"], bins=40, alpha=0.6, label=label)
ax.set_xlabel("Breidd (piksel)")
ax.set_ylabel("Tal eple")
ax.legend()
plt.show()

## Skil raudandelen åleine klassane?

In [ ]:
X_color = load_features(meta, "color")
fig, ax = plt.subplots(figsize=(7, 3.5))
for label, value in [("ripe", 1), ("unripe", 0)]:
    ax.hist(X_color.loc[y == value, "red_fraction"], bins=40, alpha=0.6, label=label)
ax.set_xlabel("Andel raude piksel")
ax.set_ylabel("Tal eple")
ax.legend()
plt.show()

## Trenings-/testsplitt per foto

In [ ]:
train_idx, test_idx = group_split(y, groups)
print(f"Trening: {len(train_idx)} eple frå {len(set(groups[train_idx]))} foto, {y[train_idx].mean():.0%} modne")
print(f"Test:    {len(test_idx)} eple frå {len(set(groups[test_idx]))} foto, {y[test_idx].mean():.0%} modne")
print("Felles foto:", len(set(groups[train_idx]) & set(groups[test_idx])))

# Del 2: Resultat frå `train.py`

In [ ]:
comparison = pd.read_csv(REPORTS / "model_comparison.csv")
comparison.round(3)

In [ ]:
metrics = json.loads((REPORTS / "metrics.json").read_text(encoding="utf-8"))
print(f"Modell:   {metrics['model']}  ({metrics['feature_set']})")
print(f"Param:    {metrics['best_params']}")
print(f"CV F1:    {metrics['cv_f1']:.3f}")
print(f"Terskel:  {metrics['threshold']:.3f}")
t = metrics["test_at_threshold"]
print(f"Test:     precision(moden) {t['moden']['precision']:.3f}, "
      f"recall(moden) {t['moden']['recall']:.3f}, accuracy {t['accuracy']:.3f}")

In [ ]:
for png in sorted(REPORTS.glob("*.png")):
    img = Image.open(png)
    plt.figure(figsize=(img.width / 150, img.height / 150))
    plt.imshow(img)
    plt.axis("off")
    plt.title(png.name, fontsize=9)
    plt.show()